##### Initial Setup

Load Standard Modules

In [ ]:
Import sys
sys.path.append("../")

import statistics
from matplotlib import pyplot as plt

Load Tudatpy Modules

In [ ]:
# Import doptrack-estimate functions
from propagation_functions.environment import *
from propagation_functions.propagation import *
from estimation_functions.estimation import *
from estimation_functions.observations_data import *

from propagation_functions.environment import *
from propagation_functions.propagation import *
from estimation_functions.estimation import *
from estimation_functions.observations_data import *

from utility_functions.time import *
from utility_functions.tle import *
from utility_functions.data import extract_tar


Load Tudatpy Modules

In [ ]:
from tudatpy import constants
from tudatpy.astro import element_conversion, frame_conversion
from tudatpy.interface import spice
from tudatpy.dynamics import environment
from tudatpy.dynamics import parameters
from tudatpy.estimation import estimation_analysis

Extract Data

In [ ]:
extract_tar("./metadata.tar.xz")
extract_tar("./data.tar.xz")

Define Import Folders

In [ ]:
metadata_folder = 'metadata/'
data_folder = 'data/'

Upload Data

In [ ]:
metadata = ['Delfi-C3_32789_202004011044.yml', 'Delfi-C3_32789_202004011219.yml',
            'Delfi-C3_32789_202004021953.yml', 'Delfi-C3_32789_202004022126.yml',
            'Delfi-C3_32789_202004031031.yml', 'Delfi-C3_32789_202004031947.yml',
            'Delfi-C3_32789_202004041200.yml',

            'Delfi-C3_32789_202004061012.yml', 'Delfi-C3_32789_202004062101.yml',
            'Delfi-C3_32789_202004072055.yml', 'Delfi-C3_32789_202004072230.yml',
            'Delfi-C3_32789_202004081135.yml']


data = ['Delfi-C3_32789_202004011044.csv', 'Delfi-C3_32789_202004011219.csv',
        'Delfi-C3_32789_202004021953.csv', 'Delfi-C3_32789_202004022126.csv',
        'Delfi-C3_32789_202004031031.csv', 'Delfi-C3_32789_202004031947.csv',
        'Delfi-C3_32789_202004041200.csv',

        'Delfi-C3_32789_202004061012.csv', 'Delfi-C3_32789_202004062101.csv',
        'Delfi-C3_32789_202004072055.csv', 'Delfi-C3_32789_202004072230.csv',
        'Delfi-C3_32789_202004081135.csv']

##### Specifying which metadata and data files should be loaded

<small> so basically what the indices does is it selects the specific metadata and data files. <br> the metadata files. saves info on the satellite, tle,time,etc <br> the csv file stores the doppler data</small>

In [1]:
def meta_and_data_selector(int: indices_remove):
    '''
    function selects which data and meta-data files to select
    returns: indices_files_to_load
    '''
    indices_files_to_load = [0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11]
    indices_file_to_load = np.delete(indices_file_to_load, indices_remove)
    return indices_files_to_load

    case_1 = meta_and_data_selector(0)

##### Setting up an initial orbit Determination

In [ ]:
def ini_orbit_det(int: indices_remove):

    indices_files_to_load = [0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11]
    indices_file_to_load = np.delete(indices_file_to_load, indices_remove)

    initial_epoch, initial_state_teme, b_star = get_tle_initial_conditions(metadata_folder + metadata[0], old_yml=False)

    # Define the propagation time, and compute the final and mid-propagation epochs accordingly.
    propagation_time = 10.0 * constants.JULIAN_DAY
    final_epoch = get_start_next_day(initial_epoch) + propagation_time
    mid_epoch = (initial_epoch + final_epoch) / 2.0

    # Retrieve the spacecraft's initial state at mid-epoch from the TLE orbit
    initial_state = propagate_sgp4(metadata_folder + metadata[0], initial_epoch, [mid_epoch], old_yml=False)[0, 1:]

    # Retrieve recording starting times
    recording_start_times = extract_recording_start_times_yml(metadata_folder, [metadata[i] for i in indices_files_to_load], old_yml=False)

    # Load and process observations
    passes_start_times, passes_end_times, observation_times, observations_set = load_and_format_observations(
        "Delfi", data_folder, [data[i] for i in indices_files_to_load], recording_start_times, old_obs_format=False)

##### Setting up the estimation arcs

In [ ]:
def arc_est(pass_option = 'per_day'):
    arc_start_times, arc_mid_times, arc_end_times = define_arcs(pass_option, passes_start_times, passes_end_times)
    print('arc_start_times', arc_start_times)
    print('arc_end_times', arc_end_times)

Setting the estimation settings

In [ ]:
def estimation_settings():
    # Define propagation_functions environment
    mass = 2.2
    ref_area = (4 * 0.3 * 0.1 + 2 * 0.1 * 0.1) / 4  # Average projection area of a 3U CubeSat
    srp_coef = 1.2
    drag_coef = 1.2
    bodies = define_environment(mass, ref_area, drag_coef, srp_coef, "Delfi", multi_arc_ephemeris=False)

    # Define accelerations exerted on Delfi
    # Warning: point_mass_gravity and spherical_harmonic_gravity accelerations should not be defined simultaneously for a single body
    accelerations = dict(
        Sun={
            'point_mass_gravity': True,
            'solar_radiation_pressure': True
        },
        Moon={
            'point_mass_gravity': True
        },
        Earth={
            'point_mass_gravity': False,
            'spherical_harmonic_gravity': True,
            'drag': True
        },
        Venus={
            'point_mass_gravity': True
        },
        Mars={
            'point_mass_gravity': True
        },
        Jupiter={
            'point_mass_gravity': True
        }
    )

    # Propagate dynamics and retrieve Delfi's initial state at the start of each arc
    orbit = propagate_initial_state(initial_state, initial_epoch, final_epoch, bodies, accelerations, "Delfi")
    arc_wise_initial_states = get_initial_states(bodies, arc_mid_times, "Delfi")


    # Redefine environment to allow for multi-arc dynamics propagation_functions
    bodies = define_environment(mass, ref_area, drag_coef, srp_coef, "Delfi", multi_arc_ephemeris=True)

    # Define multi-arc propagator settings
    multi_arc_propagator_settings = define_multi_arc_propagation_settings(arc_wise_initial_states, arc_start_times, arc_end_times,
                                                                          bodies, accelerations, "Delfi")
    # Create the DopTrack station
    define_doptrack_station(bodies)

    # Define default observation settings
    # Specify on which time interval the observation bias(es) should be defined. This will change throughout the assignment (can be 'per_pass', 'per_arc', 'global')
    # Noting that the arc duration can vary (see arc definition)
    bias_definition = 'per_pass'
    Doppler_models = dict(
        constant_absolute_bias={
            'activated': True,
            'time_interval': bias_definition
        },
        linear_absolute_bias={
            'activated': True,
            'time_interval': bias_definition
        }
    )
    observation_settings = define_observation_settings("Delfi", Doppler_models, passes_start_times, arc_start_times)

    # Define parameters to estimate
    parameters_list = dict(
        initial_state={
            'estimate': True
        },
        constant_absolute_bias={
            'estimate': True
        },
        linear_absolute_bias={
            'estimate': True
        }
    )
    parameters_to_estimate = define_parameters(parameters_list, bodies, multi_arc_propagator_settings, "Delfi",
                                               arc_start_times, arc_mid_times, [(get_link_ends_id("DopTrackStation", "Delfi"), passes_start_times)], Doppler_models)
    parameters.print_parameter_names(parameters_to_estimate)

    # Create the estimator object
    estimator = estimation_analysis.Estimator(bodies, parameters_to_estimate, observation_settings, multi_arc_propagator_settings)

    # Simulate (ideal) observations
    ideal_observations = simulate_observations_from_estimator("Delfi", observation_times, estimator, bodies)


Run the estimation

In [ ]:
# Save the true parameters to later analyse the error
truth_parameters = parameters_to_estimate.parameter_vector
nb_parameters = len(truth_parameters)

# Perform estimation_functions
nb_iterations = 10
nb_arcs = len(arc_start_times)
pod_output = run_estimation(estimator, parameters_to_estimate, observations_set, nb_arcs, nb_iterations)

errors = pod_output.formal_errors
residuals = pod_output.residual_history
mean_residuals = statistics.mean(residuals[:,nb_iterations-1])
std_residuals = statistics.stdev(residuals[:,nb_iterations-1])

residuals_per_pass = get_residuals_per_pass(observation_times, residuals, passes_start_times)

print('--------------------------------------------------------------')
for i in range(len(residuals_per_pass)):
    print('size residuals current pass', np.shape(residuals_per_pass[i]))